# Real-time Object Detection Benchmark — Ready-to-Play UI

Notebook này được tổ chức theo workflow **one-click sau khi Run All** với 4 model:

| Key | Model | Backend |
|---|---|---|
| `yolo26m` | YOLO26m | Ultralytics / PyTorch |
| `rfdetr_m` | RF-DETR-M | Hugging Face Transformers / PyTorch |
| `dfine_m` | D-FINE-M | Hugging Face Transformers / PyTorch |
| `lwdetr_m` | LW-DETR-M | Hugging Face Transformers / PyTorch |

## Workflow
1. Chạy **Run All**.
2. Notebook tự kiểm tra dependency, tải/cache checkpoint và chạy **smoke test inference** cho từng model.
3. **UI chỉ xuất hiện khi toàn bộ model đã sẵn sàng**.
4. Sau khi UI xuất hiện: chỉ cần chọn **Model → nguồn vào → Play**.

> Lần chạy đầu cần Internet để tải package/checkpoint. Các lần sau ưu tiên cache local. PyTorch/CUDA của kernel được xem là dependency nền tảng và không bị notebook tự thay đổi.


## Session 1 — Workspace và thư mục quản lý

Cell này xác định project root và tạo toàn bộ thư mục làm việc. Notebook có thể được mở từ thư mục `testmodel` hoặc từ thư mục cha của nó.

Cấu trúc chính:

```text
testmodel/
├─ cache/
│  ├─ python-packages/
│  ├─ huggingface/
│  ├─ models/
│  ├─ sources/
│  └─ tmp/
├─ data/videos/
└─ outputs/live_sessions/
```

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path


HERE = Path.cwd().resolve()
NOTEBOOK_NAMES = {
    "benchmark_object_detection.ipynb",
}
ROOT = next(
    (
        candidate
        for candidate in (HERE, HERE / "testmodel")
        if any((candidate / name).exists() for name in NOTEBOOK_NAMES)
    ),
    None,
)

if ROOT is None:
    raise RuntimeError(
        "Run this notebook from the testmodel directory or from its parent."
    )

CACHE = ROOT / "cache"
PACKAGES = CACHE / "python-packages"
HF = CACHE / "huggingface"
MODELS = CACHE / "models"
SOURCES = CACHE / "sources"
TMP = CACHE / "tmp"
VIDEOS = ROOT / "data" / "videos"
SESSIONS = ROOT / "outputs" / "live_sessions"

MANAGED_DIRS = (
    CACHE,
    PACKAGES,
    HF,
    MODELS,
    SOURCES,
    TMP,
    VIDEOS,
    SESSIONS,
)

for directory in MANAGED_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")


Project root: C:\Users\DELL\Desktop\VLM\testmodel


## Session 2 — Cache và dependency bootstrap

Notebook tự quản lý các dependency ứng dụng trong `cache/python-packages` để giảm thao tác thủ công.

Các dependency chính được chuẩn bị tự động:
- Ultralytics;
- Transformers + Hugging Face Hub;
- OpenCV + ipywidgets;

> **PyTorch không được tự cài/ghi đè**, vì bản PyTorch phải khớp driver/CUDA của máy. Nếu `torch.cuda.is_available()` là `False`, notebook vẫn có thể chạy nhưng không còn đúng mục tiêu benchmark GPU realtime.


In [2]:
import importlib.util

# Packages are installed once in the Python kernel environment, outside this notebook.
# Model checkpoints and Hugging Face/Torch runtime caches remain local to testmodel.
os.environ.update(
    {
        "HF_HOME": str(HF),
        "HUGGINGFACE_HUB_CACHE": str(HF / "hub"),
        "TRANSFORMERS_CACHE": str(HF / "hub"),
        "TORCH_HOME": str(CACHE / "torch"),
        "ULTRALYTICS_HOME": str(CACHE / "ultralytics"),
        "XDG_CACHE_HOME": str(CACHE / "xdg"),
        "TEMP": str(TMP),
        "TMP": str(TMP),
        "TMPDIR": str(TMP),
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
        "TQDM_DISABLE": "1",
    }
)

REQUIRED_MODULES = (
    "torch", "torchvision", "ultralytics", "transformers", "huggingface_hub",
    "ipywidgets", "cv2", "PIL", "yaml", "scipy", "shapely",
    "terminaltables", "typeguard", "visualdl", "pycocotools",
)
missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Missing normal Python packages: " + ", ".join(missing))

import transformers
major_minor = tuple(map(int, transformers.__version__.split(".")[:2]))
if major_minor < (5, 1):
    raise RuntimeError("Transformers >= 5.1 is required for RF-DETR-M and LW-DETR-M.")

print("Dependencies are installed in the normal Python environment.")
print("Model/cache root:", CACHE)


Dependencies are installed in the normal Python environment.
Model/cache root: C:\Users\DELL\Desktop\VLM\testmodel\cache


## Session 3 — Runtime, logging và GPU

Khởi tạo các thư viện dùng chung, logging, device và dtype. Nếu CUDA khả dụng, inference PyTorch dùng `float16`; nếu không sẽ fallback về CPU/`float32`.

In [3]:
import asyncio
import concurrent.futures
import csv
import json
import logging
import shutil
import time
import urllib.request
from contextlib import contextmanager
from dataclasses import dataclass

import cv2
import ipywidgets as widgets
import numpy as np
import torch
from IPython.display import display
from PIL import Image


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
LOG = logging.getLogger("four-model-lab")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
VIDEO_SUFFIXES = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch dtype: {DTYPE}")


Device: cuda
GPU: NVIDIA RTX A3000 12GB Laptop GPU
PyTorch dtype: torch.float16


## Session 4 — Model registry và cấu trúc kết quả

`CONFIG` là registry duy nhất của 4 model. Mỗi model có backend và nguồn checkpoint riêng. Hai dataclass `Det` và `Result` chuẩn hóa output để UI có thể xử lý tất cả model theo cùng một interface.

In [4]:
CONFIG = {
    "yolo26m": {
        "name": "YOLO26m",
        "type": "yolo",
        "weight": MODELS / "yolo26m.pt",
    },
    "rfdetr_m": {
        "name": "RF-DETR-M",
        "type": "hf",
        "repo": "Roboflow/rf-detr-medium",
    },
    "dfine_m": {
        "name": "D-FINE-M",
        "type": "hf",
        "repo": "ustc-community/dfine-medium-coco",
    },
    "lwdetr_m": {
        "name": "LW-DETR-M",
        "type": "hf",
        "repo": "AnnaZhang/lwdetr_medium_60e_coco",
    },

}

assert len(CONFIG) == 4


@dataclass
class Det:
    label: str
    score: float
    x1: float
    y1: float
    x2: float
    y2: float


@dataclass
class Result:
    detections: list[Det]
    ms: float


## Session 5 — Video và visualization utilities

Các hàm dùng chung cho:
- tìm video local;
- đọc metadata cơ bản;
- đổi working directory tạm thời;
- vẽ bounding box và confidence lên frame.

In [5]:
def videos() -> list[Path]:
    """Return all supported local video files under the managed video directory."""
    return sorted(
        path
        for path in VIDEOS.rglob("*")
        if path.suffix.lower() in VIDEO_SUFFIXES
    )


def video_text(path: Path) -> str:
    """Build a compact label for the video selector."""
    capture = cv2.VideoCapture(str(path))
    fps = capture.get(cv2.CAP_PROP_FPS) or 0
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    capture.release()

    duration = frame_count / fps if fps else 0
    return f"{path.name} | {width}x{height} | {duration:.1f}s"


@contextmanager
def cwd(path: Path):
    """Temporarily change the process working directory."""
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def paint(frame: np.ndarray, detections: list[Det]) -> np.ndarray:
    """Draw detections on a copy of the source frame."""
    output = frame.copy()

    for det in detections:
        x1, y1, x2, y2 = map(int, (det.x1, det.y1, det.x2, det.y2))
        color = (45, 220, 120)

        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            output,
            f"{det.label} {det.score:.2f}",
            (x1, max(18, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2,
        )

    return output


## Session 6 — Model adapters

Mọi backend đều triển khai cùng hai thao tác:

- `prepare(...)`: tải/cache checkpoint và load model;
- `predict(frame, conf)`: nhận một frame OpenCV và trả về `Result`.

Nhờ adapter pattern, phần UI không cần biết model phía dưới là Ultralytics, Hugging Face.

### 6.1 Base adapter + YOLO26m

In [6]:
class Adapter:
    def __init__(self, key: str):
        self.key = key
        self.c = CONFIG[key]
        self.model = None

    def unload(self) -> None:
        self.model = None
        if DEVICE == "cuda":
            torch.cuda.empty_cache()


class YOLOAdapter(Adapter):
    def prepare(self, progress) -> None:
        from ultralytics import YOLO

        weight = self.c["weight"]
        weight.parent.mkdir(parents=True, exist_ok=True)

        if not weight.exists():
            progress.note("Downloading YOLO26m into testmodel/cache/models")
            with cwd(weight.parent):
                YOLO(weight.name)

        self.model = YOLO(str(weight))
        self.model.to(DEVICE)

    def predict(self, frame: np.ndarray, conf: float) -> Result:
        started = time.perf_counter()
        prediction = self.model.predict(
            frame,
            conf=conf,
            imgsz=640,
            device=0 if DEVICE == "cuda" else "cpu",
            verbose=False,
        )[0]

        detections: list[Det] = []
        if prediction.boxes is not None:
            for box in prediction.boxes:
                x1, y1, x2, y2 = box.xyxy[0].detach().cpu().tolist()
                class_id = int(box.cls[0])
                detections.append(
                    Det(
                        label=str(prediction.names[class_id]),
                        score=float(box.conf[0]),
                        x1=x1,
                        y1=y1,
                        x2=x2,
                        y2=y2,
                    )
                )

        elapsed_ms = (time.perf_counter() - started) * 1000
        return Result(detections=detections, ms=elapsed_ms)


### 6.2 Hugging Face adapters — RF-DETR-M, D-FINE-M, LW-DETR-M

Ba model này dùng chung `AutoImageProcessor` và `AutoModelForObjectDetection`.

Trong preflight, checkpoint được tải một lần và đường dẫn snapshot local được lưu vào registry. Khi bấm **Play**, model được load lại **từ cache local**, không cần tải checkpoint lần nữa.


In [7]:
class HFAdapter(Adapter):
    def prepare(self, progress) -> None:
        from huggingface_hub import snapshot_download
        from transformers import AutoImageProcessor, AutoModelForObjectDetection

        snapshot = self.c.get("snapshot")
        if snapshot and Path(snapshot).exists():
            snapshot_path = Path(snapshot)
            progress.note(f"Loading cached snapshot: {self.c['repo']}")
        else:
            progress.note(f"Caching {self.c['repo']} locally")
            snapshot_path = Path(
                snapshot_download(
                    repo_id=self.c["repo"],
                    cache_dir=str(HF / "hub"),
                    local_files_only=False,
                )
            )
            self.c["snapshot"] = str(snapshot_path)

        self.processor = AutoImageProcessor.from_pretrained(
            snapshot_path,
            local_files_only=True,
            use_fast=False,
        )

        try:
            self.model = AutoModelForObjectDetection.from_pretrained(
                snapshot_path,
                local_files_only=True,
                dtype=DTYPE,
            )
        except TypeError:
            self.model = AutoModelForObjectDetection.from_pretrained(
                snapshot_path,
                local_files_only=True,
                torch_dtype=DTYPE,
            )

        self.model = self.model.to(DEVICE).eval()
        self.input_dtype = next(self.model.parameters()).dtype

    def predict(self, frame: np.ndarray, conf: float) -> Result:
        started = time.perf_counter()

        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        processed = self.processor(images=image, return_tensors="pt")
        inputs = {
            key: (
                value.to(DEVICE, dtype=self.input_dtype)
                if value.is_floating_point()
                else value.to(DEVICE)
            )
            for key, value in processed.items()
        }

        with torch.inference_mode():
            output = self.model(**inputs)

        raw = self.processor.post_process_object_detection(
            output,
            target_sizes=torch.tensor([image.size[::-1]], device=DEVICE),
            threshold=conf,
        )[0]

        labels = self.model.config.id2label
        detections = [
            Det(
                label=labels.get(int(class_id), str(int(class_id))),
                score=float(score),
                x1=float(box[0]),
                y1=float(box[1]),
                x2=float(box[2]),
                y2=float(box[3]),
            )
            for score, class_id, box in zip(
                raw["scores"].detach().cpu(),
                raw["labels"].detach().cpu(),
                raw["boxes"].detach().cpu().tolist(),
            )
        ]

        elapsed_ms = (time.perf_counter() - started) * 1000
        return Result(detections=detections, ms=elapsed_ms)


## Session 7 — Adapter registry và model manager

`Manager` đảm bảo chỉ một model được giữ trong bộ nhớ tại một thời điểm. Khi đổi model, model cũ được unload trước khi model mới được load.

In [8]:
ADAPTERS = {
    "yolo26m": YOLOAdapter,
    "rfdetr_m": HFAdapter,
    "dfine_m": HFAdapter,
    "lwdetr_m": HFAdapter,
}


class Manager:
    def __init__(self):
        self.key = None
        self.a = None

    def load(self, key: str, progress=None) -> None:
        if self.key == key and self.a is not None:
            return

        self.unload()
        self.a = ADAPTERS[key](key)

        LOG.info("Loading model: %s", key)
        self.a.prepare(progress or Progress())
        self.key = key
        LOG.info("Model loaded: %s", key)

    def predict(self, key: str, frame: np.ndarray, conf: float) -> Result:
        self.load(key)
        return self.a.predict(frame, conf)

    def unload(self) -> None:
        if self.a is not None:
            self.a.unload()

        self.key = None
        self.a = None


MANAGER = Manager()


## Session 8 — Strict model preflight + smoke test

Preflight không chỉ kiểm tra checkpoint có tải/load được hay không mà còn chạy **một inference thật trên frame thử** cho từng model.

Một model chỉ được đánh dấu `ready` khi hoàn thành cả:
1. dependency/backend import;
2. checkpoint/source cache;
3. load model;
4. forward/predict smoke test.

Kết quả được lưu tại `cache/model_preflight.json`.

Mặc định notebook chạy ở **strict mode**: nếu bất kỳ model nào fail, UI sẽ **không** xuất hiện. Điều này đảm bảo rằng khi UI đã xuất hiện thì các lựa chọn trong dropdown đều đã được kiểm tra end-to-end.


In [9]:
class Progress:
    def __init__(self):
        self.bar = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description="Overall",
        )
        self.sub = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description="Task",
        )
        self.text = widgets.HTML("Waiting.")
        self.view = widgets.VBox(
            [
                widgets.HTML("<b>Preparing and smoke-testing four models</b>"),
                self.bar,
                self.sub,
                self.text,
            ]
        )

    def note(self, text: str) -> None:
        self.text.value = f"<code>{text}</code>"

    def task(self, percent: int, text: str) -> None:
        self.sub.value = percent
        self.note(text)


def smoke_frame() -> np.ndarray:
    """Create a deterministic 640x640 frame for end-to-end adapter verification."""
    frame = np.full((640, 640, 3), 128, dtype=np.uint8)
    cv2.rectangle(frame, (120, 140), (300, 430), (210, 210, 210), -1)
    cv2.circle(frame, (460, 300), 90, (80, 80, 80), -1)
    return frame


def prepare_all(progress: Progress) -> dict:
    report = {}
    model_count = len(ADAPTERS)
    test_frame = smoke_frame()

    for index, key in enumerate(ADAPTERS, start=1):
        progress.bar.value = int((index - 1) * 100 / model_count)
        progress.sub.value = 0
        progress.note(f"[{index}/{model_count}] Preparing {CONFIG[key]['name']}")
        print(f"[{index}/{model_count}] Preparing: {CONFIG[key]['name']}")

        try:
            MANAGER.load(key, progress)
            progress.task(90, f"Smoke-testing {CONFIG[key]['name']}")
            result = MANAGER.predict(key, test_frame, 0.25)
            report[key] = {
                "state": "ready",
                "detail": "Checkpoint, load and smoke inference verified",
                "smoke_ms": round(result.ms, 3),
            }
            print(
                f"[{index}/{model_count}] Ready: {CONFIG[key]['name']} "
                f"({result.ms:.1f} ms smoke test)"
            )
        except Exception as exc:
            report[key] = {
                "state": "failed",
                "detail": f"{type(exc).__name__}: {exc}",
            }
            print(f"[{index}/{model_count}] Failed: {type(exc).__name__}: {exc}")
        finally:
            MANAGER.unload()

    preflight_path = CACHE / "model_preflight.json"
    preflight_path.write_text(
        json.dumps({"models": report}, indent=2),
        encoding="utf-8",
    )

    progress.bar.value = 100
    progress.sub.value = 100

    failed = [
        CONFIG[key]["name"]
        for key, state in report.items()
        if state["state"] != "ready"
    ]

    if failed:
        progress.note("Preflight failed: " + ", ".join(failed))
    else:
        progress.note("All four models are ready. Building UI...")

    return report


## Session 9 — Ready-to-play UI

Sau khi strict preflight thành công, UI chính chỉ yêu cầu ba thao tác:

1. chọn **Model**;
2. chọn **nguồn vào**: **Video file** hoặc **Live webcam**;
3. chọn video (nếu dùng video file) rồi bấm **Play**.

Video tệp được phát theo FPS gốc khi model đủ nhanh. Nếu inference chậm hơn video, UI ưu tiên hiển thị đủ frame thay vì tua nhanh. Webcam chạy trực tiếp từ camera đã chọn; bấm **Stop** sẽ dừng session, đóng camera và vẫn lưu CSV cho các frame đã xử lý.

`Confidence`, `Frame stride`, refresh, unload và thêm đường dẫn video được chuyển vào mục **Advanced**; bạn có thể bỏ qua hoàn toàn khi chạy bình thường.

Mỗi session hoàn thành hoặc bị dừng sau khi đã xử lý frame sẽ ghi CSV vào `outputs/live_sessions/`.


In [10]:
EXECUTOR = concurrent.futures.ThreadPoolExecutor(max_workers=1)


class Lab:
    def __init__(self, ready_keys: list[str]):
        self.ready_keys = ready_keys

        self.model = widgets.Dropdown(
            options=[(CONFIG[key]["name"], key) for key in ready_keys],
            description="Model:",
            value=ready_keys[0] if ready_keys else None,
            layout=widgets.Layout(width="360px"),
        )
        self.source = widgets.ToggleButtons(
            options=[("Video file", "video"), ("Live webcam", "camera")],
            value="video",
            description="Input:",
            tooltips=["Use a video stored on this computer", "Use webcam 0 directly"],
        )
        self.video = widgets.Dropdown(
            options=[(video_text(path), str(path)) for path in videos()],
            description="Video:",
            layout=widgets.Layout(width="760px"),
        )
        self.video_row = widgets.VBox([self.video])
        self.camera_index = widgets.BoundedIntText(
            value=0,
            min=0,
            max=9,
            description="Camera #:",
            layout=widgets.Layout(width="190px"),
        )
        self.camera_row = widgets.HBox(
            [
                self.camera_index,
                widgets.HTML(
                    "<small>Uses the selected local webcam only while Play is active.</small>"
                ),
            ]
        )
        self.camera_row.layout.display = "none"

        self.play = widgets.Button(
            description="Play",
            button_style="success",
            icon="play",
        )
        self.stop = widgets.Button(
            description="Stop",
            button_style="warning",
            icon="stop",
            disabled=True,
        )

        self.conf = widgets.FloatSlider(
            value=0.35,
            min=0.05,
            max=0.95,
            step=0.05,
            description="Confidence",
        )
        self.stride = widgets.IntSlider(
            value=1,
            min=1,
            max=8,
            description="Frame stride",
        )
        self.refresh = widgets.Button(description="Refresh videos")
        self.unload = widgets.Button(description="Unload model")
        self.path = widgets.Text(
            description="Local path:",
            layout=widgets.Layout(width="650px"),
        )
        self.add = widgets.Button(description="Use path")

        self.bar = widgets.IntProgress(
            min=0,
            max=100,
            description="Video",
            layout=widgets.Layout(width="760px"),
        )
        self.status = widgets.HTML("<b>All models prepared. Ready.</b>")
        self.metrics = widgets.HTML()
        self.image = widgets.Image(
            format="jpeg",
            layout=widgets.Layout(width="760px"),
        )

        self.running = False
        self.rows = []
        self.capture = None
        self.stop_event = None

        self.play.on_click(lambda _: asyncio.create_task(self.run()))
        self.stop.on_click(lambda _: self.stop_session())
        self.source.observe(self.change_source, names="value")
        self.refresh.on_click(lambda _: self.reload())
        self.unload.on_click(lambda _: MANAGER.unload())
        self.add.on_click(lambda _: self.add_path())

        advanced_body = widgets.VBox(
            [
                widgets.HBox([self.conf, self.stride]),
                widgets.HBox([self.refresh, self.unload]),
                widgets.HBox([self.path, self.add]),
            ]
        )
        advanced = widgets.Accordion(children=[advanced_body])
        advanced.set_title(0, "Advanced (optional)")
        advanced.selected_index = None

        self.view = widgets.VBox(
            [
                widgets.HTML(
                    "<h3>Interactive object detection benchmark</h3>"
                    "<div style='color:#166534'><b>READY</b> — "
                    "choose Model + input, then press Play.</div>"
                ),
                self.model,
                self.source,
                self.video_row,
                self.camera_row,
                widgets.HBox([self.play, self.stop]),
                advanced,
                self.bar,
                self.status,
                self.metrics,
                self.image,
                widgets.HTML(
                    "<small>Source videos are never overwritten. "
                    "Metrics CSV files are saved to outputs/live_sessions.</small>"
                ),
            ]
        )

    def show(self) -> None:
        display(self.view)

    def msg(self, text: str, error: bool = False) -> None:
        color = "#b91c1c" if error else "#166534"
        self.status.value = f"<b style='color:{color}'>{text}</b>"

    def change_source(self, change: dict) -> None:
        is_camera = change["new"] == "camera"
        self.video_row.layout.display = "none" if is_camera else ""
        self.camera_row.layout.display = "" if is_camera else "none"
        self.bar.description = "Camera" if is_camera else "Video"
        self.bar.value = 0
        self.bar.bar_style = "info" if is_camera else ""
        if is_camera:
            self.msg("Live webcam selected. Choose Camera # if required, then press Play.")
        else:
            self.msg("Local video selected. Playback follows the source FPS when possible.")

    def stop_session(self) -> None:
        if not self.running:
            return
        self.running = False
        if self.stop_event is not None:
            self.stop_event.set()
        if self.capture is not None:
            self.capture.release()
        self.stop.disabled = True
        self.msg("Stop requested. Closing the input after the current inference finishes.")

    def reload(self) -> None:
        current = self.video.value
        options = [(video_text(path), str(path)) for path in videos()]
        self.video.options = options
        values = {value for _, value in options}
        if current in values:
            self.video.value = current
        self.msg(f"Found {len(options)} local videos.")

    def add_path(self) -> None:
        path = Path(self.path.value).expanduser()
        if path.exists() and path.suffix.lower() in VIDEO_SUFFIXES:
            resolved = path.resolve()
            existing = list(self.video.options)
            if str(resolved) not in {value for _, value in existing}:
                existing.append((video_text(resolved), str(resolved)))
            self.video.options = existing
            self.video.value = str(resolved)
            self.msg("Added local video; the file was not copied.")
        else:
            self.msg("Enter an existing local video path.", True)

    def open_capture(self, is_live: bool):
        if not is_live:
            path = Path(self.video.value)
            capture = cv2.VideoCapture(str(path))
            return capture, path, path.stem

        index = int(self.camera_index.value)
        backend = getattr(cv2, "CAP_DSHOW", None)
        capture = (
            cv2.VideoCapture(index, backend)
            if backend is not None
            else cv2.VideoCapture(index)
        )
        if not capture.isOpened() and backend is not None:
            capture.release()
            capture = cv2.VideoCapture(index)
        return capture, None, f"webcam_{index}"

    async def wait_for_video_clock(self, deadline: float) -> bool:
        """Wait until the next source frame is due, but make Stop immediate."""
        delay = deadline - time.perf_counter()
        if delay <= 0:
            await asyncio.sleep(0)
            return self.running
        try:
            await asyncio.wait_for(self.stop_event.wait(), timeout=delay)
        except asyncio.TimeoutError:
            return self.running
        return False

    def render(self, frame: np.ndarray) -> None:
        ok, encoded = cv2.imencode(".jpg", frame)
        if ok:
            self.image.value = encoded.tobytes()

    async def run(self) -> None:
        if self.running:
            return
        if self.source.value == "video" and not self.video.value:
            self.msg(
                "No video selected. Put a video in data/videos or use Advanced > Local path.",
                True,
            )
            return
        if not self.model.value:
            self.msg("No prepared model is available.", True)
            return

        self.running = True
        self.rows = []
        self.stop_event = asyncio.Event()
        self.play.disabled = True
        self.stop.disabled = False
        self.model.disabled = True
        self.source.disabled = True
        self.video.disabled = True
        self.camera_index.disabled = True

        key = self.model.value
        is_live = self.source.value == "camera"
        event_loop = asyncio.get_running_loop()
        capture = None
        path = None

        try:
            self.msg(f"Loading {CONFIG[key]['name']} from local cache...")
            await event_loop.run_in_executor(EXECUTOR, MANAGER.load, key)
            if not self.running:
                return

            capture, path, source_name = self.open_capture(is_live)
            self.capture = capture
            if not capture.isOpened():
                source_hint = f"webcam {self.camera_index.value}" if is_live else str(path)
                raise RuntimeError(f"Cannot open {source_hint}")

            total_frames = 0 if is_live else int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
            source_fps = capture.get(cv2.CAP_PROP_FPS) or 0.0
            frame_period = 1 / source_fps if source_fps > 0 else 1 / 30
            source_fps_text = (
                f"{source_fps:.1f}" if source_fps > 0 else "device default"
            )
            frame_index = 0
            begun = time.perf_counter()
            next_frame_at = begun
            self.bar.description = "Camera" if is_live else "Video"
            self.bar.value = 0
            self.bar.bar_style = "info" if is_live else ""

            while self.running:
                if not is_live and not await self.wait_for_video_clock(next_frame_at):
                    break

                ok, frame = capture.read()
                if not ok:
                    break
                if not self.running:
                    break

                frame_index += 1
                if not is_live:
                    next_frame_at += frame_period
                if (frame_index - 1) % self.stride.value:
                    continue

                result = await event_loop.run_in_executor(
                    EXECUTOR,
                    MANAGER.predict,
                    key,
                    frame,
                    self.conf.value,
                )
                if not self.running:
                    break

                self.render(paint(frame, result.detections))
                self.rows.append(
                    {
                        "frame": frame_index,
                        "objects": len(result.detections),
                        "inference_ms": round(result.ms, 3),
                        "model": key,
                        "input": source_name,
                    }
                )

                average_ms = sum(
                    row["inference_ms"] for row in self.rows
                ) / len(self.rows)
                elapsed = time.perf_counter() - begun
                effective_fps = len(self.rows) / elapsed if elapsed else 0.0
                if is_live:
                    timing = (
                        f"<b>Input:</b> Webcam {self.camera_index.value} | "
                        f"<b>Capture FPS:</b> {source_fps_text} | "
                        f"<b>Effective:</b> {effective_fps:.1f} FPS"
                    )
                    self.bar.value = 0
                else:
                    eta_seconds = max(
                        0.0,
                        (total_frames - frame_index)
                        * average_ms
                        / self.stride.value
                        / 1000,
                    )
                    timing = (
                        f"<b>Frame:</b> {frame_index}/{total_frames or '?'} | "
                        f"<b>Source FPS:</b> {source_fps:.1f} | "
                        f"<b>Effective:</b> {effective_fps:.1f} FPS | "
                        f"<b>ETA:</b> {eta_seconds:.1f}s"
                    )
                    self.bar.value = (
                        int(frame_index * 100 / total_frames) if total_frames else 0
                    )

                self.metrics.value = (
                    f"{timing} | <b>Objects:</b> {len(result.detections)} | "
                    f"<b>Inference:</b> {result.ms:.1f} ms | "
                    f"<b>Average:</b> {average_ms:.1f} ms | "
                    f"<b>Elapsed:</b> {elapsed:.1f}s"
                )

            stopped = not self.running
            if self.rows:
                output_path = SESSIONS / f"{int(time.time())}_{key}_{source_name}.csv"
                with output_path.open("w", newline="", encoding="utf-8") as file:
                    writer = csv.DictWriter(
                        file,
                        fieldnames=self.rows[0].keys(),
                    )
                    writer.writeheader()
                    writer.writerows(self.rows)
                action = "Session stopped" if stopped else "Session completed"
                self.msg(f"{action}. Metrics: {output_path}")
            elif stopped:
                self.msg("Session stopped before any frames were processed.")
            elif is_live:
                self.msg("Webcam stream ended without processed frames.", True)
            else:
                self.msg("Video ended without processed frames.", True)

        except Exception as exc:
            LOG.exception("session failed")
            self.msg(f"Session failed: {type(exc).__name__}: {exc}", True)

        finally:
            if capture is not None:
                capture.release()
            self.capture = None
            self.stop_event = None
            self.running = False
            self.play.disabled = False
            self.stop.disabled = True
            self.model.disabled = False
            self.source.disabled = False
            self.video.disabled = False
            self.camera_index.disabled = False


## Session 10 — Strict launch gate

`Run All` luôn chuẩn bị và smoke-test tất cả 4 model trước.

- Nếu **4/4 ready** → UI được tạo.
- Nếu **bất kỳ model nào failed** → notebook dừng tại preflight và **không tạo UI giả-ready**.

Nhờ đó, khi bạn nhìn thấy UI, workflow còn lại đúng là: **Model → Video → Play**.


In [11]:
PREPARE_ON_RUN_ALL = True
REQUIRE_ALL_MODELS_READY = True

PROGRESS = Progress()
display(PROGRESS.view)

REPORT = prepare_all(PROGRESS) if PREPARE_ON_RUN_ALL else {}
READY_KEYS = [
    key
    for key in CONFIG
    if REPORT.get(key, {}).get("state") == "ready"
]
FAILED_KEYS = [key for key in CONFIG if key not in READY_KEYS]

if REQUIRE_ALL_MODELS_READY and FAILED_KEYS:
    details = "<br>".join(
        f"<b>{CONFIG[key]['name']}</b>: {REPORT.get(key, {}).get('detail', 'unknown error')}"
        for key in FAILED_KEYS
    )
    display(
        widgets.HTML(
            "<div style='border:1px solid #b91c1c;padding:12px'>"
            f"<b style='color:#b91c1c'>UI NOT STARTED — preflight is not {len(CONFIG)}/{len(CONFIG)} ready.</b><br>"
            f"{details}<br><br>"
            "Fix the environment/backend error above, then restart kernel and Run All."
            "</div>"
        )
    )
    raise RuntimeError(
        "Strict preflight failed. UI is intentionally not created until all four models pass."
    )

LAB = Lab(READY_KEYS)
LAB.show()


2026-09-27 13:59:01,769 | INFO | Loading model: yolo26m


[1/4] Preparing: YOLO26m


2026-09-27 13:59:03,234 | INFO | Model loaded: yolo26m
2026-09-27 13:59:12,055 | INFO | Loading model: rfdetr_m


[1/4] Ready: YOLO26m (8760.1 ms smoke test)
[2/4] Preparing: RF-DETR-M


2026-09-27 13:59:15,143 | INFO | HTTP Request: GET https://huggingface.co/api/models/Roboflow/rf-detr-medium/revision/main "HTTP/1.1 200 OK"
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
[transformers] Requested pil backend is not available. Falling back to torchvision backend.
2026-09-27 13:59:15,814 | INFO | Model loaded: rfdetr_m
2026-09-27 13:59:16,433 | INFO | Loading model: dfine_m


[2/4] Ready: RF-DETR-M (610.3 ms smoke test)
[3/4] Preparing: D-FINE-M


2026-09-27 13:59:16,712 | INFO | HTTP Request: GET https://huggingface.co/api/models/ustc-community/dfine-medium-coco/revision/main "HTTP/1.1 200 OK"
2026-09-27 13:59:17,386 | INFO | Model loaded: dfine_m
2026-09-27 13:59:19,039 | INFO | Loading model: lwdetr_m


[3/4] Ready: D-FINE-M (1643.7 ms smoke test)
[4/4] Preparing: LW-DETR-M


2026-09-27 13:59:19,310 | INFO | HTTP Request: GET https://huggingface.co/api/models/AnnaZhang/lwdetr_medium_60e_coco/revision/main "HTTP/1.1 200 OK"
2026-09-27 13:59:19,912 | INFO | Model loaded: lwdetr_m


[4/4] Ready: LW-DETR-M (89.7 ms smoke test)


## Session 11 — Cách sử dụng

### Sau khi mở notebook
1. Chọn kernel có **PyTorch + CUDA** hoạt động.
2. Bấm **Run All** một lần.
3. Lần đầu, chờ notebook tải dependency/checkpoint và smoke-test 4 model.
4. Khi UI có trạng thái **READY**, chỉ làm ba việc:
   - chọn **Model**;
   - chọn **nguồn vào**: **Video file** hoặc **Live webcam**;
   - bấm **Play**.

### Không cần làm thủ công sau khi UI đã hiện
- không cần tải checkpoint;
- không cần bấm prepare model;
- không cần chạy cell riêng cho model;
- không cần reload checkpoint;
- không cần cấu hình confidence/stride nếu dùng giá trị mặc định.

### Advanced
Mục `Advanced (optional)` chỉ dành cho trường hợp muốn:
- đổi confidence;
- bỏ frame bằng `Frame stride`;
- thêm video ngoài `data/videos/`;
- unload GPU thủ công.

### Output
Mỗi session hoàn thành ghi CSV vào `outputs/live_sessions/` với:
- frame index;
- số object;
- inference latency;
- model key.

> Lưu ý kỹ thuật: model được smoke-test trước UI nhưng không giữ cả 4 model đồng thời trong VRAM. Khi bấm **Play**, model đã chọn được load lại từ **cache local**; thao tác này tự động và không yêu cầu bạn làm gì thêm.
